# XP Exercises: Flower Classification using CNN

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os
import warnings
warnings.filterwarnings("ignore")

print("TensorFlow version:", tf.__version__)
tf.random.set_seed(42)

# Reduced resolution as recommended for faster training
HEIGHT = 48
WIDTH = 48
BATCH_SIZE = 32
NUM_CLASSES = 14

FLOWER_CLASSES = [
    "Astilbe", "Bellflower", "Black-eyed Susan", "Calendula",
    "California Poppy", "Carnation", "Common Daisy", "Coreopsis",
    "Dandelion", "Iris", "Rose", "Sunflower", "Tulip", "Water Lily"
]

# TODO: Set the path to your dataset directory after downloading from Kaggle
TRAIN_DIR = "flowers_dataset/train"
VAL_DIR = "flowers_dataset/val"

## Part 1: Data Exploration and Visualization

In [ ]:
# Load the dataset using image_dataset_from_directory
train_dataset = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=(HEIGHT, WIDTH),
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=True,
    seed=42
)

val_dataset = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=(HEIGHT, WIDTH),
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False
)

class_names = train_dataset.class_names
print("Classes found:", class_names)
print("Number of classes:", len(class_names))

In [ ]:
# Print number of images per class
def count_images_per_class(directory):
    counts = {}
    for class_name in sorted(os.listdir(directory)):
        class_path = os.path.join(directory, class_name)
        if os.path.isdir(class_path):
            counts[class_name] = len(os.listdir(class_path))
    return counts


train_counts = count_images_per_class(TRAIN_DIR)
val_counts = count_images_per_class(VAL_DIR)

counts_df = pd.DataFrame({
    "Train": train_counts,
    "Validation": val_counts
}).fillna(0).astype(int)

print(counts_df)
print(f"\nTotal training images: {counts_df['Train'].sum()}")
print(f"Total validation images: {counts_df['Validation'].sum()}")

In [ ]:
counts_df.plot(kind="bar", figsize=(14, 5), color=["steelblue", "tomato"])
plt.title("Number of Images per Class")
plt.xlabel("Flower Species")
plt.ylabel("Image Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
def visualize_images(dataset, class_names, images_per_class=9):
    """
    Displays a 3x3 grid of images for each class in the dataset.
    """
    # Collect images grouped by class
    images_by_class = {name: [] for name in class_names}

    for images, labels in dataset:
        label_indices = tf.argmax(labels, axis=1).numpy()
        for img, label_idx in zip(images.numpy(), label_indices):
            class_name = class_names[label_idx]
            if len(images_by_class[class_name]) < images_per_class:
                images_by_class[class_name].append(img)

        if all(len(v) >= images_per_class for v in images_by_class.values()):
            break

    for class_name, imgs in images_by_class.items():
        if len(imgs) == 0:
            continue
        fig, axes = plt.subplots(3, 3, figsize=(6, 6))
        for i, ax in enumerate(axes.flatten()):
            if i < len(imgs):
                ax.imshow(imgs[i].astype("uint8"))
            ax.axis("off")
        fig.suptitle(class_name, fontsize=14, fontweight="bold")
        plt.tight_layout()
        plt.show()


visualize_images(train_dataset, class_names)

**Analysis — Anticipated Challenges**

- **Similar colors**: Several species (Calendula, Common Daisy, Coreopsis, Sunflower) share yellow/orange petal tones, which can confuse a model relying heavily on color features.
- **Similar shapes**: Daisy-like flowers (Common Daisy, Black-eyed Susan, California Poppy) have radially symmetric petals around a central disc, making shape alone insufficient to distinguish them.
- **Intra-class variation**: A single species can appear in many colors (e.g., Tulip, Carnation), different growth stages, and different lighting conditions, increasing within-class variance.
- **Background clutter**: Outdoor photos often include leaves, soil, or other flowers in the background, adding noise that a CNN must learn to ignore.
- **Class imbalance**: With far more training images (13,642) than validation images (98), and likely uneven distribution across the 14 classes, the model may be biased toward more frequent classes.

## Part 2: Model Architecture Design

In [ ]:
def build_cnn_model(height, width, num_classes):
    """
    Builds a CNN for multi-class flower classification.

    Architecture rationale:
    - 4 convolutional blocks with increasing filter counts (32 -> 64 -> 128 -> 128)
      let the network learn increasingly abstract features (edges -> textures -> petal shapes).
    - Batch Normalization after each conv layer stabilizes training and allows higher
      learning rates, which is especially helpful with the small 48x48 input resolution.
    - MaxPooling after each block progressively reduces spatial dimensions, controlling
      the parameter count given the already-low input resolution.
    - Dropout (0.3-0.5) in the dense layers combats overfitting, important here since
      training set (13,642) is much larger than validation set (98), risking optimistic
      validation metrics that may not generalize to truly unseen data.
    - GlobalAveragePooling instead of Flatten reduces parameter count significantly
      compared to a large Flatten + Dense combination, which matters for a 14-class
      problem with limited validation data to tune against.
    """
    model = keras.Sequential([
        layers.Input(shape=(height, width, 3)),
        layers.Rescaling(1.0 / 255),

        layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(128, (3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(128, (3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.GlobalAveragePooling2D(),

        layers.Dense(128, activation="relu"),
        layers.Dropout(0.4),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.3),

        layers.Dense(num_classes, activation="softmax")
    ])
    return model


model = build_cnn_model(HEIGHT, WIDTH, NUM_CLASSES)
model.summary()

**Justification of architectural choices**

- **4 convolutional blocks**: Provides enough depth to capture hierarchical features (petal edges, textures, overall shape) without being excessive for 48x48 images, where too many pooling layers would shrink the feature maps to nearly nothing.
- **Increasing filter counts (32→64→128→128)**: Early layers need fewer filters to detect simple patterns (edges, color gradients); deeper layers need more filters to represent the larger number of possible complex feature combinations.
- **Batch Normalization**: Normalizes activations between layers, reducing internal covariate shift and allowing the network to train faster and more reliably.
- **Global Average Pooling instead of Flatten**: Drastically reduces the number of parameters feeding into the dense layers, which is critical for reducing overfitting risk given the small validation set.
- **Dropout (0.3-0.4)**: Randomly deactivates neurons during training, forcing the network to learn redundant, robust representations rather than memorizing the training set.

## Part 3: Hyperparameter Tuning

In [ ]:
def compile_and_train(model, train_ds, val_ds, optimizer, epochs=20, callbacks=None):
    model.compile(
        optimizer=optimizer,
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
        callbacks=callbacks or [],
        verbose=1
    )
    return history


early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)

lr_scheduler = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6
)

print("Training utilities defined.")

In [ ]:
# Experiment table — track results across hyperparameter configurations
experiments = [
    {"name": "Adam, lr=0.001, batch=32",   "optimizer": keras.optimizers.Adam(learning_rate=0.001)},
    {"name": "Adam, lr=0.0005, batch=32",  "optimizer": keras.optimizers.Adam(learning_rate=0.0005)},
    {"name": "RMSprop, lr=0.001, batch=32","optimizer": keras.optimizers.RMSprop(learning_rate=0.001)},
    {"name": "SGD+momentum, lr=0.01",      "optimizer": keras.optimizers.SGD(learning_rate=0.01, momentum=0.9)},
]

experiment_results = []

for exp in experiments:
    print(f"\n{'='*60}\nRunning: {exp['name']}\n{'='*60}")
    exp_model = build_cnn_model(HEIGHT, WIDTH, NUM_CLASSES)
    history = compile_and_train(
        exp_model, train_dataset, val_dataset,
        optimizer=exp["optimizer"],
        epochs=10,
        callbacks=[early_stop, lr_scheduler]
    )
    best_val_acc = max(history.history["val_accuracy"])
    best_val_loss = min(history.history["val_loss"])
    experiment_results.append({
        "Configuration": exp["name"],
        "Best Val Accuracy": round(best_val_acc, 4),
        "Best Val Loss": round(best_val_loss, 4)
    })

results_df = pd.DataFrame(experiment_results).sort_values("Best Val Accuracy", ascending=False)
print("\n\nHyperparameter Experiment Summary:")
print(results_df.to_string(index=False))

**Best configuration**: The configuration with the highest validation accuracy in the table above is selected as the best hyperparameter setting. Adam generally converges faster and more reliably than plain SGD on image classification tasks, and a moderate learning rate (0.001) typically balances convergence speed with training stability.

## Part 4: Data Augmentation

In [ ]:
train_augmentation = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=False,   # Flowers rarely appear upside-down in real photos
    fill_mode="nearest"
)

val_augmentation = ImageDataGenerator(rescale=1.0 / 255)

train_aug_generator = train_augmentation.flow_from_directory(
    TRAIN_DIR,
    target_size=(HEIGHT, WIDTH),
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

val_aug_generator = val_augmentation.flow_from_directory(
    VAL_DIR,
    target_size=(HEIGHT, WIDTH),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

In [ ]:
# Visualize augmented samples
sample_batch, _ = next(train_aug_generator)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i, ax in enumerate(axes.flatten()):
    if i < len(sample_batch):
        ax.imshow(sample_batch[i])
    ax.axis("off")
plt.suptitle("Sample Augmented Training Images", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
best_optimizer = keras.optimizers.Adam(learning_rate=0.001)

model_augmented = build_cnn_model(HEIGHT, WIDTH, NUM_CLASSES)
model_augmented.compile(
    optimizer=best_optimizer,
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history_augmented = model_augmented.fit(
    train_aug_generator,
    validation_data=val_aug_generator,
    epochs=25,
    callbacks=[early_stop, lr_scheduler],
    verbose=1
)

**Most effective augmentations for this dataset**

- **Rotation and horizontal flip** are highly effective because flowers can naturally appear at any angle and orientation in photographs, and this augmentation directly addresses that real-world variability without distorting flower anatomy.
- **Zoom** helps the model become invariant to the flower's distance from the camera, which varies significantly across the dataset's photos.
- **Vertical flip** was disabled because flowers in nature almost always grow upward — training on upside-down flowers introduces unrealistic patterns the model would never see in production use.
- **Shear and shift** provide mild robustness to off-center framing but are used with smaller magnitudes since aggressive shearing can distort delicate petal structures that are key for species discrimination.

## Part 5: Performance Evaluation and Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(history_augmented.history["loss"], label="Train Loss")
axes[0].plot(history_augmented.history["val_loss"], label="Val Loss")
axes[0].set_title("Loss over Epochs")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(history_augmented.history["accuracy"], label="Train Accuracy")
axes[1].plot(history_augmented.history["val_accuracy"], label="Val Accuracy")
axes[1].set_title("Accuracy over Epochs")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()

plt.suptitle("Training History — Augmented Model", fontsize=13)
plt.tight_layout()
plt.show()

**Overfitting/underfitting analysis**: If training accuracy continues to climb while validation accuracy plateaus or declines, and training loss keeps falling while validation loss rises, this indicates overfitting — the model is memorizing training examples rather than learning generalizable features. Given the very small validation set (98 images across 14 classes, roughly 7 per class), validation metrics here should be interpreted cautiously since they have high variance. If both training and validation accuracy remain low throughout training, this would indicate underfitting, suggesting the model architecture is too simple or training has not run long enough.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Get true labels and predictions on the validation set
val_aug_generator.reset()
y_true = val_aug_generator.classes
y_pred_probs = model_augmented.predict(val_aug_generator, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

class_labels = list(val_aug_generator.class_indices.keys())

print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_labels))

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(11, 9))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_labels, yticklabels=class_labels)
plt.title("Confusion Matrix — Flower Classification")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

**Difficult classes**: Based on the confusion matrix, the species most often confused with each other are typically those sharing similar colors or radial petal structures — for example, Common Daisy vs Black-eyed Susan, or Calendula vs Coreopsis vs Sunflower (all yellow, daisy-like flowers). Species with more visually distinctive features, such as Water Lily (aquatic, floating leaves) or Iris (distinctive elongated petal shape), tend to be classified more reliably.

In [ ]:
# Visualize predictions on a sample of test/validation images
val_aug_generator.reset()
sample_images, sample_labels = next(val_aug_generator)
sample_preds = model_augmented.predict(sample_images, verbose=0)

fig, axes = plt.subplots(3, 3, figsize=(12, 12))
for i, ax in enumerate(axes.flatten()):
    if i < len(sample_images):
        ax.imshow(sample_images[i])
        true_label = class_labels[np.argmax(sample_labels[i])]
        pred_label = class_labels[np.argmax(sample_preds[i])]
        correct = true_label == pred_label
        color = "green" if correct else "red"
        ax.set_title(f"True: {true_label}\nPred: {pred_label}",
                      color=color, fontsize=9)
    ax.axis("off")

plt.suptitle("Model Predictions on Validation Samples\n(Green = Correct, Red = Misclassified)",
             fontsize=13)
plt.tight_layout()
plt.show()

**Misclassification analysis**: Misclassified examples often involve flowers photographed at unusual angles, partially occluded by leaves or other flowers, or photographed in lighting conditions (strong shadows, overexposure) that distort their typical color profile. Close-up macro shots that crop out the overall flower shape can also mislead the model, since it then must rely solely on petal texture and color, which are less discriminative features.

## Part 6 (Optional): Model Saving and Deployment

In [ ]:
# Save the trained model
model_augmented.save("flower_classification_model.h5")
print("Model saved as flower_classification_model.h5")

# Also save in the recommended Keras format
model_augmented.save("flower_classification_model.keras")
print("Model saved as flower_classification_model.keras")

**Deployment possibilities**

- **Flask/Django web app**: Load the saved `.h5`/`.keras` model in a Flask route that accepts an uploaded image, preprocesses it to 48x48 RGB, runs `model.predict()`, and returns the predicted flower species as JSON or rendered HTML.
- **Cloud deployment**: The model can be containerized with Docker and deployed to Google Cloud Run, AWS Lambda (with a container image), or AWS SageMaker for scalable real-time inference via a REST API endpoint.
- **Mobile app**: Convert the model to TensorFlow Lite (`tf.lite.TFLiteConverter`) for on-device inference in an Android or iOS app, enabling offline flower identification directly from the phone's camera.